# Hypothetical Resume Embedding (HYRE) — ConFit v2

This notebook:
1. Converts your `resume-job-description-fit` data into ConFit v2 format
2. Generates hypothetical resumes via DeepSeek-R1-Distill-Qwen-32B
3. Produces the final job CSV with HYRE for training

Label mapping: `Good Fit` → 1, `No Fit` / `Potential Fit` → 0

## 0. Colab Setup

Clone the ConFit v2 repo, install dependencies, and upload your data.

In [ ]:
# ── Clone repo (pick ONE option) ──────────────────────────────────────

# !git clone https://github.com/<your-username>/ConFit-v2.git /content/ConFit-v2

REPO_ROOT = "/content/ConFit-v2"  # <-- adjust if your path differs

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────
# The repo's requirements.txt has old pinned versions that conflict with Colab.
# Install only what's needed for training:
!pip install -q deepspeed pytorch-lightning torchmetrics wandb transformers \
    jieba nltk pandas scikit-learn tqdm xgboost faiss-gpu-cu12

In [ ]:
# ── Upload your data ───────────────────────────────────────────────────
# Put train.csv and test.csv into the repo. Adjust paths if you upload elsewhere.
import os
DATA_DIR = os.path.join(REPO_ROOT, "resume_job_fit_raw")
os.makedirs(DATA_DIR, exist_ok=True)

# Option A: Upload manually via Colab file browser to DATA_DIR

# Option B: Upload interactively
# from google.colab import files
# uploaded = files.upload()  # select train.csv and test.csv
# for name, data in uploaded.items():
#     with open(os.path.join(DATA_DIR, name), "wb") as f:
#         f.write(data)

# Option C: If already on Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/drive/MyDrive/path/to/train.csv {DATA_DIR}/
# !cp /content/drive/MyDrive/path/to/test.csv {DATA_DIR}/

print("Files in data dir:", os.listdir(DATA_DIR))

## 1. Imports

In [ ]:
import os, json, random, re, hashlib
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# Only needed if you use a gated model (e.g. jina-embeddings-v2-base-zh).
# Get your token at https://huggingface.co/settings/tokens
# login(token="hf_...")

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────
TRAIN_CSV  = os.path.join(REPO_ROOT, "resume_job_fit_raw/train.csv")
TEST_CSV   = os.path.join(REPO_ROOT, "resume_job_fit_raw/test.csv")
OUTPUT_DIR = os.path.join(REPO_ROOT, "dataset/resume_job_fit")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1. Convert to ConFit v2 format

Your data has `resume_text`, `job_description_text`, `label` (string).  
ConFit v2 needs:
- `all_resume.csv` — deduplicated resumes with `user_id`, `text_resume`
- `all_job.csv` — deduplicated jobs with `jd_no`, `text_job`
- Label JSONL files with `user_id`, `jd_no`, `satisfied` (0 or 1)

In [ ]:
train_raw = pd.read_csv(TRAIN_CSV)
test_raw  = pd.read_csv(TEST_CSV)
all_raw = pd.concat([train_raw, test_raw], ignore_index=True)

print(f"Combined: {len(all_raw):,} rows")
print(f"\nLabel distribution:\n{all_raw['label'].value_counts()}")

In [ ]:
def text_to_id(text: str) -> str:
    """Deterministic ID from text content via SHA-256."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def convert_label(label_str: str) -> int:
    """Good Fit → 1, everything else → 0."""
    return 1 if label_str == "Good Fit" else 0


all_raw["user_id"]   = all_raw["resume_text"].apply(text_to_id)
all_raw["jd_no"]     = all_raw["job_description_text"].apply(text_to_id)
all_raw["satisfied"] = all_raw["label"].apply(convert_label)

print(f"Binary label distribution:\n{all_raw['satisfied'].value_counts()}")
print(f"Unique resumes: {all_raw['user_id'].nunique()}")
print(f"Unique jobs:    {all_raw['jd_no'].nunique()}")

In [ ]:
# ── Deduplicated resume & job CSVs ───────────────────────────────────
resume_df = (
    all_raw[["user_id", "resume_text"]]
    .drop_duplicates(subset="user_id")
    .rename(columns={"resume_text": "text_resume"})
)
job_df = (
    all_raw[["jd_no", "job_description_text"]]
    .drop_duplicates(subset="jd_no")
    .rename(columns={"job_description_text": "text_job"})
)

resume_df.to_csv(os.path.join(OUTPUT_DIR, "all_resume.csv"), index=False)
job_df.to_csv(os.path.join(OUTPUT_DIR, "all_job.csv"), index=False)

print(f"Unique resumes: {len(resume_df):,}")
print(f"Unique jobs:    {len(job_df):,}")

In [ ]:
# ── Split combined data into train / valid / test (70/15/15) ─────────
all_labels = all_raw[["user_id", "jd_no", "satisfied"]]

train_labels, temp_labels = train_test_split(
    all_labels, test_size=0.3, random_state=SEED, stratify=all_labels["satisfied"]
)
valid_labels, test_labels = train_test_split(
    temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels["satisfied"]
)

# Save as JSONL
for name, df_labels in [("train_labeled_data", train_labels),
                         ("valid_classification_data", valid_labels),
                         ("test_classification_data", test_labels)]:
    path = os.path.join(OUTPUT_DIR, f"{name}.jsonl")
    df_labels.to_json(path, orient="records", lines=True)
    pos = (df_labels["satisfied"] == 1).sum()
    print(f"{name}: {len(df_labels):,} pairs ({pos:,} positive)")

In [ ]:
# ── Build ranking evaluation files ─────────────────────────────────────
# rank_job.json: for each resume, rank candidate jobs
# rank_resume.json: for each job, rank candidate resumes

def build_rank_data(labels_df):
    """Build ranking dicts from label pairs."""
    rank_job = {}   # user_id -> {pos_jd_nos, neg_jd_nos}
    rank_resume = {}  # jd_no -> {pos_user_ids, neg_user_ids}

    for _, row in labels_df.iterrows():
        uid, jid, sat = row["user_id"], row["jd_no"], row["satisfied"]
        key = "pos" if sat == 1 else "neg"

        rank_job.setdefault(uid, {"pos": [], "neg": []})[key].append(jid)
        rank_resume.setdefault(jid, {"pos": [], "neg": []})[key].append(uid)

    # Keep only entries that have at least one positive
    rank_job = {k: v for k, v in rank_job.items() if len(v["pos"]) > 0 and len(v["neg"]) > 0}
    rank_resume = {k: v for k, v in rank_resume.items() if len(v["pos"]) > 0 and len(v["neg"]) > 0}
    return rank_job, rank_resume


rank_job, rank_resume = build_rank_data(test_labels)

with open(os.path.join(OUTPUT_DIR, "rank_job.json"), "w") as f:
    json.dump(rank_job, f)
with open(os.path.join(OUTPUT_DIR, "rank_resume.json"), "w") as f:
    json.dump(rank_resume, f)

print(f"rank_job entries:    {len(rank_job)} (resumes with both pos & neg jobs)")
print(f"rank_resume entries: {len(rank_resume)} (jobs with both pos & neg resumes)")

In [ ]:
# ── Also build valid ranking files ─────────────────────────────────────
valid_rank_job, valid_rank_resume = build_rank_data(valid_labels)

with open(os.path.join(OUTPUT_DIR, "valid_rank_job.json"), "w") as f:
    json.dump(valid_rank_job, f)
with open(os.path.join(OUTPUT_DIR, "valid_rank_resume.json"), "w") as f:
    json.dump(valid_rank_resume, f)

print(f"valid_rank_job:    {len(valid_rank_job)}")
print(f"valid_rank_resume: {len(valid_rank_resume)}")

In [ ]:
# ── Summary of generated files ─────────────────────────────────────────
print("\nFiles in", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:45s} {size/1024:.1f} KB")

## 2. Create few-shot templates

We auto-build templates from "Good Fit" pairs in the training data.  
These are used as one-shot examples when prompting the LLM.

In [ ]:
TEMPLATES_PATH = os.path.join(OUTPUT_DIR, "templates.json")
NUM_TEMPLATES = 20

# Pick Good Fit pairs from training split as templates
train_raw_with_labels = all_raw.iloc[train_labels.index]
good_fit_rows = train_raw_with_labels[train_raw_with_labels["label"] == "Good Fit"]
sampled = good_fit_rows.sample(n=min(NUM_TEMPLATES, len(good_fit_rows)), random_state=SEED)

templates = []
for _, row in sampled.iterrows():
    templates.append({
        "job": row["job_description_text"],
        "resume": row["resume_text"],
    })

with open(TEMPLATES_PATH, "w", encoding="utf-8") as f:
    json.dump(templates, f, indent=2, ensure_ascii=False)

print(f"Saved {len(templates)} templates from Good Fit pairs")
print(f"Job preview:    {templates[0]['job'][:200]}...")
print(f"Resume preview: {templates[0]['resume'][:200]}...")

## 3. Generate hypothetical resumes via LLM

For each unique job, prompt DeepSeek-R1-Distill-Qwen-32B to generate an ideal resume.

In [ ]:
# ── Load model ──────────────────────────────────────────────────────────
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

# 32B in bf16 ≈ 64GB — fits on a single A100 80GB.
llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Loaded {MODEL_ID}")

In [ ]:
def build_hyre_prompt(job_text: str, template: dict) -> str:
    """Construct the prompt exactly as the ConFit v2 codebase does."""
    return "\n".join([
        "Here is a template pair of matching resume and job:",
        "[The start of the example job]",
        template["job"],
        "[The end of the example job]",
        "[The start of the example resume]",
        template["resume"],
        "[The end of the example resume]",
        "You are a helpful assistant. Following the above example pair of job and resume, "
        "construct an ideal resume for the target job shown below. You should strictly follow "
        "the format of the given pairs, make sure the resume you give perfectly matches the "
        "target job, and directly return your answer in plain texts.",
        "[The start of the target job]",
        job_text,
        "[The end of the target job]",
    ])


def clean_llm_output(text: str) -> str:
    """Strip thinking blocks (DeepSeek R1) and bracketed markers."""
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r"\[The start of .*?\]\n?", "", text)
    text = re.sub(r"\n?\[The end of .*?\]?", "", text)
    return text.strip()


def generate_hyre(prompt: str, max_new_tokens: int = 2048) -> str:
    """Run a single generation through the local model."""
    messages = [{"role": "user", "content": prompt}]
    chat_input = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    if hasattr(chat_input, "input_ids"):
        input_ids = chat_input["input_ids"].to(llm_model.device)
        attention_mask = chat_input["attention_mask"].to(llm_model.device)
    else:
        input_ids = chat_input.to(llm_model.device)
        attention_mask = None
    with torch.no_grad():
        output_ids = llm_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    generated = output_ids[0, input_ids.shape[1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True)


# Quick test with one job
sample_job = job_df.iloc[0]
prompt = build_hyre_prompt(sample_job["text_job"], random.choice(templates))
print("=== PROMPT (first 600 chars) ===")
print(prompt[:600])
print("...")

In [ ]:
# ── Generate for ALL unique jobs ───────────────────────────────────────
# Set MAX_JOBS to a small number to test first, then None for full run.
MAX_JOBS = 5  # <-- set to None to process all jobs

jobs_to_process = job_df if MAX_JOBS is None else job_df.head(MAX_JOBS)

results = {}  # jd_no -> hypothetical resume text

# Resume from checkpoint if exists
checkpoint_path = os.path.join(OUTPUT_DIR, "converted_raw.json")
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Resumed from checkpoint: {len(results)} already done")

for i, (_, row) in enumerate(tqdm(jobs_to_process.iterrows(), total=len(jobs_to_process), desc="Generating HYRE")):
    jd_no = row["jd_no"]
    if jd_no in results:
        continue  # skip already generated

    job_text = row["text_job"]
    template = random.choice(templates)
    prompt   = build_hyre_prompt(job_text, template)
    raw_output = generate_hyre(prompt)
    results[jd_no] = clean_llm_output(raw_output)

    # Checkpoint every 50 jobs
    if (i + 1) % 50 == 0:
        with open(checkpoint_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

# Final save
with open(checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nGenerated {len(results)} hypothetical resumes")

## 4. Merge & concatenate with original JD

In [ ]:
CONCAT_MODE = True  # recommended by the paper

job_text_lookup = dict(zip(job_df["jd_no"], job_df["text_job"]))

merged_rows = {"jd_no": [], "text_job": []}
for jd_no, hypo_resume in results.items():
    merged_rows["jd_no"].append(jd_no)
    if CONCAT_MODE:
        merged_rows["text_job"].append(
            job_text_lookup[jd_no] + "\n[Example resume]\n" + hypo_resume
        )
    else:
        merged_rows["text_job"].append(hypo_resume)

merged_df = pd.DataFrame(merged_rows)

suffix = "_concat" if CONCAT_MODE else ""
out_csv = os.path.join(OUTPUT_DIR, f"all_job_converted{suffix}.csv")
merged_df.to_csv(out_csv, index=False)
print(f"Saved merged job CSV ({len(merged_df)} rows) to {out_csv}")

## 5. Inspect the results

In [ ]:
idx = random.randint(0, len(merged_df) - 1)
sample = merged_df.iloc[idx]
jd_no = sample["jd_no"]

print("=" * 60)
print(f"JD_NO: {jd_no}")
print("=" * 60)

print("\n── ORIGINAL JOB ──")
print(job_text_lookup.get(jd_no, "(not found)")[:1000])

print("\n── HYPOTHETICAL RESUME (raw) ──")
print(results.get(jd_no, "(not found)")[:1000])

print("\n── FINAL text_job (what the encoder sees) ──")
print(sample["text_job"][:1500])

## 6. Train ConFit v2

Free the DeepSeek model from GPU memory first, then run the training pipeline.

In [ ]:
# ── Free the LLM from GPU memory ──────────────────────────────────────
import gc

del llm_model
del llm_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
import os
os.chdir(REPO_ROOT)
os.environ["PYTHONPATH"] = REPO_ROOT
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"Working directory: {os.getcwd()}")

In [ ]:
!python runners/trainer/train_confit.py \
  --save_path model_checkpoints/confit_v2_fit \
  --resume_data_path /content/dataset/resume_job_fit/all_resume.csv \
  --job_data_path /content/dataset/resume_job_fit/all_job_converted_concat.csv \
  --train_label_path /content/dataset/resume_job_fit/train_labeled_data.jsonl \
  --valid_label_path /content/dataset/resume_job_fit/valid_classification_data.jsonl \
  --classification_data_path /content/dataset/resume_job_fit/test_classification_data.jsonl \
  --rank_resume_data_path /content/dataset/resume_job_fit/rank_resume.json \
  --rank_job_data_path /content/dataset/resume_job_fit/rank_job.json \
  --dataset_type recruiting_data_v2 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --encode_all True \
  --embedding_method mean_pool \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --do_normalize true \
  --temperature 0.05 \
  --do_both_rj_hard_neg true \
  --train_batch_size 4 \
  --val_batch_size 4 \
  --gradient_accumulation_steps 2 \
  --precision bf16 \
  --strategy deepspeed_stage_2 \
  --max_epochs 10

## 7. Mine hard negatives

Use the trained model to find "runner-up" negatives — resumes/jobs that are close in embedding space but not actual matches. These make training much harder and more effective in the second pass.

In [ ]:
# ── Find the best checkpoint from step 6 ───────────────────────────────
import glob

ckpt_dir = "model_checkpoints/confit_v2_fit"
# Look for .fp32 checkpoints (deepspeed converted) or .ckpt files
ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.fp32"))) or \
        sorted(glob.glob(os.path.join(ckpt_dir, "*.ckpt")))
print("Available checkpoints:")
for c in ckpts:
    print(f"  {c}")

# Use the best (first .fp32) or last checkpoint
BEST_CKPT = ckpts[0] if ckpts else None
print(f"\nUsing: {BEST_CKPT}")

In [ ]:
# ── Mine hard negatives for training set ───────────────────────────────
!python src/utils/hard_negative_mining.py \
  --model_path {BEST_CKPT} \
  --file_path /content/dataset/resume_job_fit/hard_negatives \
  --file_name hard_negatives_train.json \
  --resume_data_path /content/dataset/resume_job_fit/all_resume.csv \
  --job_data_path /content/dataset/resume_job_fit/all_job_converted_concat.csv \
  --classification_data_path /content/dataset/resume_job_fit/train_labeled_data.jsonl \
  --dataset_type recruiting_data_v2 \
  --batch_size 16 \
  --seed 42 \
  --lower_bound 0.04 \
  --upper_bound 0.03 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --encode_all True \
  --embedding_method mean_pool \
  --finegrained_loss noop \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --do_normalize true \
  --temperature 0.05 \
  --do_both_rj_hard_neg true

In [ ]:
# ── Mine hard negatives for validation set ─────────────────────────────
!python src/utils/hard_negative_mining.py \
  --model_path {BEST_CKPT} \
  --file_path /content/dataset/resume_job_fit/hard_negatives \
  --file_name hard_negatives_valid.json \
  --resume_data_path /content/dataset/resume_job_fit/all_resume.csv \
  --job_data_path /content/dataset/resume_job_fit/all_job_converted_concat.csv \
  --classification_data_path /content/dataset/resume_job_fit/valid_classification_data.jsonl \
  --dataset_type recruiting_data_v2 \
  --batch_size 16 \
  --seed 42 \
  --lower_bound 0.04 \
  --upper_bound 0.03 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --encode_all True \
  --embedding_method mean_pool \
  --finegrained_loss noop \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --do_normalize true \
  --temperature 0.05 \
  --do_both_rj_hard_neg true

## 8. Re-train with hard negatives

Second training pass using the mined hard negatives for stronger contrastive learning.

In [ ]:
!python runners/trainer/train_confit.py \
  --save_path model_checkpoints/confit_v2_fit_hard_neg \
  --resume_data_path /content/dataset/resume_job_fit/all_resume.csv \
  --job_data_path /content/dataset/resume_job_fit/all_job_converted_concat.csv \
  --train_label_path /content/dataset/resume_job_fit/train_labeled_data.jsonl \
  --valid_label_path /content/dataset/resume_job_fit/valid_classification_data.jsonl \
  --classification_data_path /content/dataset/resume_job_fit/test_classification_data.jsonl \
  --rank_resume_data_path /content/dataset/resume_job_fit/rank_resume.json \
  --rank_job_data_path /content/dataset/resume_job_fit/rank_job.json \
  --hard_negative_path /content/dataset/resume_job_fit/hard_negatives/hard_negatives_train.json \
  --hard_negative_valid_path /content/dataset/resume_job_fit/hard_negatives/hard_negatives_valid.json \
  --dataset_type recruiting_data_v2 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --encode_all True \
  --embedding_method mean_pool \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --do_normalize true \
  --temperature 0.05 \
  --do_both_rj_hard_neg true \
  --train_batch_size 4 \
  --val_batch_size 4 \
  --gradient_accumulation_steps 2 \
  --precision bf16 \
  --strategy deepspeed_stage_2 \
  --max_epochs 10

In [ ]:
# Token length distribution (optional — requires jina tokenizer access)
# enc_tokenizer = AutoTokenizer.from_pretrained("jinaai/jina-embeddings-v2-base-zh")
# lengths = merged_df["text_job"].apply(lambda t: len(enc_tokenizer.encode(t)))
# print(lengths.describe())
# print(f"\nRows exceeding 3200 tokens: {(lengths > 3200).sum()} / {len(lengths)}")